# Gramáticas libres, Parsing y dependencias

Hasta ahora nos hemos enfocado en las palabras: su identificación, la forma en la que estan constituidas y le hemos asignado etiquetas para hacer explícita cierta información lingüística. Sin embargo, para explorar las complejidades de la construcción de oraciones primero un método para representar las reglas de consturcción y, despúes, una forma de librarnos de la **ambigüedad** que conllevan estas reglas.

Este *método* serán las gramáticas libres de contexto cuyo objetivo es dar una descripción explicita de un lenguaje. Una gramática libre de contexto nos va ayudar a capturar ciertos patrones en la estructura sintáctica de una oración. Con ella podremos hacer que la computadora pueda determinar si un elemento es un sujeto, un objeto directo, un objeto indirecto, un verbo, etc.

![](https://imgs.xkcd.com/comics/formal_languages_2x.png)

**[audience looks around] 'What just happened?' 'There must be some context we're missing.'**

Una gramática libre de contexto, CFG, es una cuatro-tupla 

$$CFG = (∆, Σ, d0, R)$$

donde:

- $∆ = {d0, ..., dn}$ - es un conjunto de símbolos no terminales
- $Σ$ - es un conjunto de símbolos terminales
- $d0 ∈ ∆$ - es el símbolo de inicio
- $R : ∆ → ∆ ∪ Σ$ - es un sistema de reglas

### Constituyentes principales de una oración

- **Frase nominal (FN)/Noun phrase (NP)**: El sustantivo es el núcleo. Ejemplos: la niña, el niño triste,
los perros, etc.

- **Frase verbal(FV)/verb phrase (VP)**: El verbo es el núcleo, está formado por el verbo y sus
argumentos. Ejemplo: come, come dulces, leyó los cuentos, ...

- **Frase preposicional(FP)/prepositional phrase (PP)**: El núcleo es la preposición, se combina con
sustantivos o frases nominales. Ejemplos: con el perro, abajo de la mesa, para su amigo, etc.


#### [Syntactic Categories](https://www.nltk.org/book/ch08.html#code-cfg1)

In [15]:
import nltk

In [16]:
simple_gramma = nltk.CFG.fromstring("""
S -> NP VP
S -> VP
NP -> Det N
VP -> V
VP -> V NP
Det -> 'that' | 'this' | 'a' | 'the' | 'an'
N -> 'man' | 'school' | 'flag' | 'book'
V -> 'read' | 'do' | 'look'
""")

En este caso hemos definido nuestra gramática y podemos, entonces parsear las oraciones que queramos analizar. Nótese aquí que distinguimos los símobolos terminales con comillas simples '', mientras que los símbolos no terminales no se marcan de esta forma.

In [ ]:
sentences = [
    "do the flag",
    "the man read this book",
    "man a read",
    "read this magazine"
]
# Try Recursive
parser = nltk.ChartParser(simple_gramma)
for sent in sentences:
    for tree in parser.parse(sent.split()): 
        tree.pretty_print(unicodelines=True, nodedist=4)
        print(tree)

In [18]:
grammar = nltk.CFG.fromstring("""
S -> NP VP
NP -> Det N | Det N PP
VP -> V NP | V NP PP
PP -> P NP
NP -> 'I'
N -> 'man' | 'park' | 'telescope' | 'dog'
Det -> 'the' | 'a'
P -> 'in' | 'with'
V -> 'saw'
"""
)
# Sauce: https://www.nltk.org/_modules/nltk/parse/shiftreduce.html
parser = nltk.ShiftReduceParser(grammar, trace=2)
for p in parser.parse("I saw a man in the park".split()):
    print(p)

Parsing 'I saw a man in the park'
    [ * I saw a man in the park]
  S [ 'I' * saw a man in the park]
  R [ NP * saw a man in the park]
  S [ NP 'saw' * a man in the park]
  R [ NP V * a man in the park]
  S [ NP V 'a' * man in the park]
  R [ NP V Det * man in the park]
  S [ NP V Det 'man' * in the park]
  R [ NP V Det N * in the park]
  R [ NP V NP * in the park]
  R [ NP VP * in the park]
  R [ S * in the park]
  S [ S 'in' * the park]
  R [ S P * the park]
  S [ S P 'the' * park]
  R [ S P Det * park]
  S [ S P Det 'park' * ]
  R [ S P Det N * ]
  R [ S P NP * ]
  R [ S PP * ]


## La horrenda ambigüedad

![](https://i.pinimg.com/236x/af/77/fe/af77fe576a152e4bcbf9e24762a70886--funny-ha-ha-funny-shit.jpg)

"While hunting in Africa, I shot an elephant in my pajamas. How he got into my pajamas, I don't know."
> Groucho Marx movie, Animal Crackers (1930)

In [32]:
groucho_grammar = nltk.CFG.fromstring("""
S -> NP VP
PP -> P NP
NP -> Det N | Det N PP | 'I'
VP -> V NP | VP PP
Det -> 'an' | 'my'
N -> 'elephant' | 'pajamas'
V -> 'shot'
P -> 'in'
""")

In [34]:
sent = "I shot an elephant in my pajamas"
parser = nltk.ChartParser(groucho_grammar)
trees = parser.parse(sent.split())
for tree in trees: 
    tree.pretty_print(unicodelines=True, nodedist=4)

        S                                                            
 ┌──────┴───────────────────────┐                                        
 │                              VP                                   
 │              ┌───────────────┴────────────────┐                       
 │              VP                               PP                  
 │      ┌───────┴──────┐                  ┌──────┴──────┐                
 │      │              NP                 │             NP           
 │      │       ┌──────┴────────┐         │      ┌──────┴────────┐       
 NP     V      Det              N         P     Det              N   
 │      │       │               │         │      │               │       
 I     shot     an           elephant     in     my           pajamas

        S                                                     
 ┌──────┴────────────────┐                                        
 │                       VP                                   
 │      ┌───────

- Time flies like an arrow
- La niña vio al elefante con un paraguas
- Astronomers saw stars with ears

#### PCFG

In [5]:
astro_gramma = nltk.PCFG.fromstring("""
S -> NP VP [1.0]
PP -> P NP [1.0]
VP -> V NP [0.7]
VP -> VP PP [0.3]
V -> 'saw'  [1.0]
P -> 'with' [1.0]
NP -> NP PP [0.4]
NP -> 'astronomers' [0.1]
NP -> 'ears' [0.18]
NP -> 'saw' [0.04]
NP -> 'stars' [0.18]
NP -> 'telescopes' [0.1]
""" 
)

In [6]:
print(astro_gramma)

Grammar with 12 productions (start state = S)
    S -> NP VP [1.0]
    PP -> P NP [1.0]
    VP -> V NP [0.7]
    VP -> VP PP [0.3]
    V -> 'saw' [1.0]
    P -> 'with' [1.0]
    NP -> NP PP [0.4]
    NP -> 'astronomers' [0.1]
    NP -> 'ears' [0.18]
    NP -> 'saw' [0.04]
    NP -> 'stars' [0.18]
    NP -> 'telescopes' [0.1]


In [19]:
sent = "astronomers saw stars with ears"
viterbi_parser = nltk.ViterbiParser(astro_gramma)
for tree in viterbi_parser.parse(sent.split()):
    print(tree)

(S
  (NP astronomers)
  (VP (V saw) (NP (NP stars) (PP (P with) (NP ears))))) (p=0.0009072)


In [22]:
taco_grammar = nltk.PCFG.fromstring("""
O    -> FN FV     [0.7]
O    -> FV FN     [0.3]
FN   -> Sust      [0.6]
FN   -> Det Sust  [0.4]
FV   -> V FN      [0.8]
FV   -> FN V      [0.2]
Sust -> 'Juan'    [0.5]
Sust -> 'tacos'   [0.5]
Det  -> 'unos'    [1.0]
V    -> 'come'    [1.0]
""")
viterbi_parser = nltk.ViterbiParser(taco_grammar)

In [26]:
sentences = [
    "Juan come unos tacos",
    "unos tacos Juan come"
]
for sent in sentences:
    for tree in viterbi_parser.parse(sent.split()):
        print(tree)

(O
  (FN (Sust Juan))
  (FV (V come) (FN (Det unos) (Sust tacos)))) (p=0.0336)
(O
  (FN (Det unos) (Sust tacos))
  (FV (FN (Sust Juan)) (V come))) (p=0.0084)


Dada esta gramática una construcción más probable es "Juan come unos tacos", mientras que si bien "unos tacos Juan come" es aceptable, su probabilidad es menor.


### Parseo de dependencias

Un parseo de dependencias devuelve las dependencias que se dan entre los tokens de una oración. Estas dependencias suelen darse entre pares de tokens. Esto es, que relaciones tienen las palabras con otras palabras.

#### Freeling - https://nlp.lsi.upc.edu/freeling/demo/demo.php

#### Manual

<img src="https://static.vecteezy.com/system/resources/previews/005/948/290/original/five-speed-manual-gear-shift-icon-five-gear-manual-transmission-shift-symbol-automotive-parts-free-vector.jpg" width=300>

Aquí definimos un algoritmo basado en transiciones para parsear dependencias. Se propone una serie de reglas simples para analizar de manera sencilla oraciones con estructura simple.

##### Algoritmo

Definimos un parser que considera las relaciones entre sustantivos, verbos y determinantes. Se asumen las siguientes reglas:

* Un <b>determinante</b> seguido de un <b>sustantivo</b> define una dependencia DET (determinante).
* Un <b>sustantivo</b> seguido de un <b>verbo</b> define una dependencia NSUBJ (sujeto).
* Un <b>verbo</b> seguid de un <b>sustantantivo</b> define una dependencia DOBJ (objeto directo)

In [29]:
from nltk.stem import SnowballStemmer

#Genera un stemmer para simplificar los tokens
stemmer = SnowballStemmer('spanish')

In [44]:
# Categoría de tokens
Nouns = ['gat','perr', 'niñ', 'sop']
Verbs = ['corr', 'jueg', 'com', 'salt']
Dets = ['el', 'la', 'un', 'las', 'los', 'unos']

def parse(sentence):
    """
    Función para parseo de dependencias a partir de reglas simples basadas en 
    relaciones de sustantivos, verbos y determinantes.
    
    Arguments
    ---------
    sentence : list
        Lista de tokens para parsear.
    
    Returns
    -------
    dependencies : list
        Lista de dependencias entre pares de palabras.
    """
    # Inicializa el stack
    stack = ['root']
    # Guarda tokens en buffer
    buffer = sentence.split()
    # Guarda las dependencias
    dependencies = []
    
    # Agrega primer token del buffer a stack
    new_token = buffer.pop(0)
    stack.append(new_token) 
    
    # Criterio de finalización
    final = False
    while not final:
        # Obtiene tokens en tope de stack
        w1 = stack[-1] 
        w2 = stack[-2]
        # Realiza stemming de los tokens
        s1 = stemmer.stem(w1)
        s2 = stemmer.stem(w2)
        
        # Regla LeftArrow para dependencia DET
        if s1 in Nouns and s2 in Dets:
            dependencies.append('{} <--DET {}'.format(w2,w1))
            stack.remove(w2)

        # Regla LeftArrow para dependencia NSUBJ
        elif s1 in Verbs and s2 in Nouns:
            dependencies.append('{} <--NSUBJ {}'.format(w2,w1))
            stack.remove(w2)
            
        # Regla RightArrow para dependencia OBJ
        elif s1 in Nouns and s2 in Verbs:
            dependencies.append('{} DOBJ--> {}'.format(w2,w1))
            stack.remove(w1)

        # Shift
        else:
            try:
                # Intenta obtener elemento del buffer
                new_token = buffer.pop(0)
                stack.append(new_token)
            except:
                # Si son los últimos elementos y el de la izquierda es root
                # Asignamos la dependencía de la oración
                if len(stack) == 2 and s2 == "root":
                    dependencies.append(f"root -> {s1}")
                # Si no lo logra detiene el algoritmo
                final = True
    return dependencies

In [49]:
sentence = 'la niña come la sopa'

print(" | ".join(parse(sentence)))

la <--DET niña | niña <--NSUBJ come | la <--DET sopa | come DOBJ--> sopa | root -> com


#### Auto

<img src="https://content.artofmanliness.com/uploads/2018/05/Transmission-Altogether-1.jpg" width=500>

In [55]:
import spacy
from spacy import displacy

In [53]:
!python -m spacy download es_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 19.3 MB/s eta 0:00:000:00:010:00:01:01
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')


In [54]:
nlp = spacy.load("es_core_news_md")

In [85]:
doc = nlp("La niña come un suani")

In [86]:
#options = {"compact": True, "bg": "#09a3d5", "color": "white"}
displacy.render(doc, style="dep")#, options=options)

In [87]:
for chunk in doc.noun_chunks:
    print("text::", chunk.text)
    print("root::", chunk.root.text)
    print("root dep::", chunk.root.dep_)
    print("root head::", chunk.root.head.text)
    print("="*10)

text:: La niña
root:: niña
root dep:: nsubj
root head:: come
text:: un suani
root:: suani
root dep:: obj
root head:: come


In [88]:
for token in doc:
    print("token::", token.text)
    print("dep::", token.dep_)
    print("head::", token.head.text)
    print("head POS::", token.head.pos_)
    print("CHILDS")
    print([child for child in token.children])
    print("="*10)

token:: La
dep:: det
head:: niña
head POS:: NOUN
CHILDS
[]
token:: niña
dep:: nsubj
head:: come
head POS:: VERB
CHILDS
[La]
token:: come
dep:: ROOT
head:: come
head POS:: VERB
CHILDS
[niña, suani]
token:: un
dep:: det
head:: suani
head POS:: PROPN
CHILDS
[]
token:: suani
dep:: obj
head:: come
head POS:: VERB
CHILDS
[un]


### Recursos

- https://www.nltk.org/book/ch08.html
- https://spacy.io/usage/models
- https://spacy.io/usage/linguistic-features/#dependency-parse